# 03 — Training runner (C0–C4)

Generic config-driven runner: pick a config, `train_run` does the
rest (checkpoints + auto-resume on Drive, §8.2 — if Colab
disconnects, just re-run the notebook: it resumes from last.ckpt).

Rules baked in (§0): the run is FULLY described by `configs/c*.yaml`
(never edit configs for a one-off — that's a new run); no run
outside the §8.4 budget; **the test set is evaluated once, at the
end, with the val-selected checkpoint only** (§0.7) — this notebook
only evaluates VAL. Test evaluation is a separate, deliberate act
through the logging harness.

Thin runner: all logic lives in `sharp_har/`.

## 1. Environment setup

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/FilippoIsoni/sharp-har.git"
REPO_DIR = Path("/content/sharp-har")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)

# After the clone, so the file actually exists on a fresh runtime.
!pip install -q -r {REPO_DIR}/requirements.txt

sys.path.insert(0, str(REPO_DIR))

from sharp_har.utils import set_seed, get_git_hash, read_yaml

set_seed(42)
print("git hash:", get_git_hash(REPO_DIR))

import torch
assert torch.cuda.is_available(), "training needs the GPU runtime"
print("GPU:", torch.cuda.get_device_name(0))

## 2. Mount Drive + staging

In [ ]:
import zipfile
from google.colab import drive

drive.mount("/content/drive")

paths_cfg = read_yaml(REPO_DIR / "configs" / "paths.yaml")
drive_root = Path(paths_cfg["drive_root"])
stage_dir = Path(paths_cfg["stage_dir"])

if not (stage_dir.exists() and any(stage_dir.rglob("*.txt"))):
    stage_dir.mkdir(parents=True, exist_ok=True)
    for zip_name in paths_cfg["zips"]:
        src = drive_root / zip_name
        dst = Path("/content") / zip_name
        print(f"copying {src} -> {dst}")
        subprocess.run(["cp", str(src), str(dst)], check=True)
        with zipfile.ZipFile(dst) as zf:
            zf.extractall(stage_dir)
print("staged:", stage_dir)

## 3. Pick the run

`RUN` is the config stem: `c0_sharp`, `c1_ce`, `c2_grl`,
`c3_supcon`, `c4_supcon_grl`.

In [ ]:
RUN = "c1_ce"  # <-- the only knob in this notebook

cfg = read_yaml(REPO_DIR / "configs" / f"{RUN}.yaml")
CKPT_ROOT = Path(paths_cfg["ckpt_root"])
print(f"run {cfg['name']}: backbone={cfg['backbone']}, loss={cfg['loss']['type']}, "
      f"sampler={cfg['train']['sampler']}, {cfg['train']['max_epochs']} epochs x "
      f"{cfg['train']['epoch_steps']} steps")

## 4. Train (auto-resumes from Drive if interrupted)

In [ ]:
from sharp_har.train import train_run

out = train_run(cfg, stage_dir=stage_dir, ckpt_dir=CKPT_ROOT, repo_dir=REPO_DIR)
print("run dir:", out["run_dir"])
print("best val macro-F1:", out["best_val_macro_f1"])
print((out["run_dir"] / "history.csv").read_text())

## 5. VAL evaluation (harness CSVs)

CE runs only — SupCon encoders (C3/C4) are evaluated in phase B via
`cache_features` → `linear_probe` on the grid checkpoints
(epoch40/50/60.ckpt), not here.

In [ ]:
from sharp_har.harness import evaluate

if cfg["loss"]["type"] == "ce":
    metrics_csv = evaluate(
        out["run_dir"] / "best.ckpt", REPO_DIR / cfg["split_file"], "val",
        stage_dir=stage_dir, repo_dir=REPO_DIR,
        labels=cfg.get("labels"),
    )
    import pandas as pd
    print(pd.read_csv(metrics_csv).to_string())
else:
    print("SupCon run: phase-B probe selection, nothing to evaluate here (§6-C3)")

## ⚠️ Test evaluation — NOT part of this notebook

§0.7: one test evaluation per stream, at the END, with the
val-selected checkpoint, decided by the team. When that moment
comes, use `harness.evaluate(...)` / `evaluate_c0(...)` with
`set_name="test"` — every invocation is logged to
`test_invocations.jsonl`. Do not add that cell here casually.